# 5.1. Exam question: Feature Selection

#### *Gianluca Bontempi, Cédric Simar, and Martin Colot - materials from Jacopo De Stefani and Theo Verhelst*


## General imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession

## Data generation

In [ ]:
def genData(N, n, random_seed):
    np.random.seed(random_seed)

    #Inputs and the weights of the linear combination are drawn at random
    X = np.random.rand(N, n)
    theta = np.random.rand(n)
    noise = np.random.rand(N)

    Y = np.dot(X, theta) + noise
    Y = Y[:, np.newaxis]
    Z = np.concatenate((Y, X), axis=1)

    print("Number of observations :", N)
    print("Number of features :", n)

    print("Dimension of X :", X.shape)
    print("Dimension of theta :", theta.shape)
    print("Dimension of Y :", Y.shape)

    return (X, Y, Z, theta)

In [ ]:
X, Y, Z, theta = genData(100, 300, 42)

Number of observations : 100
Number of features : 300
Dimension of X : (100, 300)
Dimension of theta : (300,)
Dimension of Y : (100, 1)


In [ ]:
data = Z
data = np.concatenate((np.arange(len(data)).reshape(1, -1).transpose(), data), axis=1)

In [ ]:
data.shape

(100, 302)

In [ ]:
pd.DataFrame(data).head()

,0,1,2,3,4,5,6,7,8,9,...,292,293,294,295,296,297,298,299,300,301
0,0.0,75.135860,0.548814,0.715189,0.602763,0.544883,0.423655,0.645894,0.437587,0.891773,...,0.368585,0.136900,0.822118,0.189848,0.511319,0.224317,0.097844,0.862192,0.972919,0.960835
1,1.0,71.586198,0.906555,0.774047,0.333145,0.081101,0.407241,0.232234,0.132488,0.053427,...,0.739884,0.898062,0.672582,0.528940,0.304446,0.997962,0.362189,0.470649,0.378245,0.979527
2,2.0,75.124183,0.174658,0.327988,0.680349,0.063208,0.607249,0.477647,0.284000,0.238413,...,0.441321,0.486410,0.448369,0.567846,0.621169,0.498180,0.866789,0.627735,0.401428,0.416692
3,3.0,75.295814,0.810839,0.348192,0.211455,0.059383,0.876027,0.918546,0.120120,0.334474,...,0.816151,0.911451,0.276337,0.369524,0.379894,0.560451,0.668218,0.286717,0.019462,0.399222
4,4.0,79.700210,0.308528,0.942185,0.888265,0.860311,0.653000,0.344289,0.548849,0.815225,...,0.378268,0.657862,0.359453,0.900367,0.983275,0.030427,0.193623,0.112250,0.042364,0.227741


## Spark Session

In [ ]:
# Initialize Spark Session (local mode)
# Start Spark session with local master and 2 cores
spark = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("LinearRegression") \
    .getOrCreate()

# Let us retrieve the sparkContext object
sc = spark.sparkContext

spark.conf.set("spark.sql.shuffle.partitions", "2") # reduce number of partitions to run on google Colab

In [ ]:
#Let us use 2 partitions
B = 2
Z_RDD = sc.parallelize(data, B).cache()

In [ ]:
np.array(Z_RDD.collect()).shape

(100, 302)

**Exercise:**

Let us consider a feature selection problem with a single output target Y and n input variables $X_1, \dots, X_n$. Suppose that the dataset (made of $N$ observations) is stored in a vertical format, i.e. each line corresponds to a sample with X and Y. Also, the first column contains the ID of the line.

\begin{equation}
\begin{pmatrix}
  0       & X_{1,1}   & X_{1,2}   & \cdots  & X_{1,n} & Y_1  \\
  1       & X_{2,1}   & X_{2,2}   & \cdots  & X_{2,n} & Y_2 \\
  \cdots \\
  N       & X_{N,1}   & X_{N,2}   & \cdots  & X_{N,n} & Y_N  \\
\end{pmatrix}
\end{equation}


Write a map-reduce code (RDD-based) to perform forward selection where at each step the selection aims to minimize the linear least-squares residual error.

Note that
- We make the hypothesis that the number $n$ is huge and that the number of features to select $k << n$
- We make the hypothesis that each least-squares computation can be computed in a memory-resident manner (i.e. no distributed computation is required for such step)

**Your Solution**

**Solution**

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [ ]:
N = 100 # samples
n = 300 # features
k = 10 # features to select


selected_features = []
all_features = [i+1 for i in range(n)]


for _ in range(k):
    candidates = list(set(all_features) - set(selected_features))
    selected_features_bc = sc.broadcast(selected_features) # broadcast ids of selected features to all workers

    # 1. create new samples with already selected features, each candidate feature to select, and label
    a = Z_RDD.flatMap(lambda x : [(i, [x[j] for j in selected_features_bc.value] + [x[i], x[-1]]) for i in candidates])

    # 2. Group the samples with respect to the condidate feature
    b = a.groupByKey()

    # 3. Compute MSE for each condidate feature
    def MSE_mapper(data):
        id, data = data
        x = [d[:-1] for d in data]
        y = [d[-1] for d in data]
        mse = mean_squared_error(LinearRegression().fit(x, y).predict(x), y)
        return id, mse

    c = b.map(MSE_mapper)

    # 4. Select candidate feature with lowest MSE
    best_feature, best_mse = c.reduce(lambda a, b: a if a[1] < b[1] else b)

    # 5. Add best feature to selection
    selected_features.append(best_feature)
    selected_features_bc.unpersist()

    print(f"Selected feature: {best_feature} with MSE: {best_mse}")

Selected feature: 177 with MSE: 0.0733844489787447
Selected feature: 287 with MSE: 0.0668864938102409
Selected feature: 41 with MSE: 0.06065250546348559
Selected feature: 133 with MSE: 0.055747035067795554
Selected feature: 113 with MSE: 0.05103168938412119
Selected feature: 160 with MSE: 0.04605155402517945
Selected feature: 12 with MSE: 0.04268688739427988
Selected feature: 244 with MSE: 0.03978410021594435
Selected feature: 183 with MSE: 0.0369859394449483
Selected feature: 285 with MSE: 0.034492910760668544


In [ ]:
selected_features

[177, 287, 41, 133, 113, 160, 12, 244, 183, 285]